In [ ]:
import pandas as pd

# 1. 加载整个 2.xlsx 文件
xls = pd.ExcelFile('/kaggle/input/datasets/ericlin073233/program/2.xlsx')

# 2. 打印真实表名
print("2.xlsx 里的真实表名：", xls.sheet_names)

# 3. 按位置读取（绝不出错）
df_load = pd.read_excel(xls, sheet_name=1, header=0)  # 第 1 张表
df_pv = pd.read_excel(xls, sheet_name=0, header=0)    # 第 0 张表

# 4. 读取 1.xlsx
df_price = pd.read_excel('/kaggle/input/datasets/ericlin073233/program/1.xlsx', header=0)

# 5. 打印列名和行数
print("负载列名：", df_load.columns.tolist())
print("光伏列名：", df_pv.columns.tolist())
print("负载行数：", len(df_load))
print("光伏行数：", len(df_pv))

# 6. 直接提取数据
prices = df_price['电价'].values.astype(float)
loads = df_load.iloc[:, -1].values.astype(float) # 取最后一列数据
pvs = df_pv.iloc[:, -1].values.astype(float)    # 取最后一列数据

print(f"时间点数量: {len(prices)}")

In [24]:
import pandas as pd
import pulp

# 1. 读取数据（ 修正：sheet_name 对调！）
xls = pd.ExcelFile('/kaggle/input/datasets/ericlin073233/program/2.xlsx')
df_load = pd.read_excel(xls, sheet_name=0, header=0)  # 第 1 张表是负载
df_pv = pd.read_excel(xls, sheet_name=1, header=0)    # 第 2 张表是光伏
df_price_wave = pd.read_excel('/kaggle/input/datasets/ericlin073233/program/4.xlsx', header=0)

# 统一日期格式
for df in [df_load, df_pv, df_price_wave]:
    df['日期'] = pd.to_datetime(df.iloc[:, 0]).dt.strftime('%Y-%m-%d')

# 选一天有数据的日期
date_str = '2025-04-15'
load_row = df_load[df_load['日期'] == date_str]
pv_row = df_pv[df_pv['日期'] == date_str]
price_row = df_price_wave[df_price_wave['日期'] == date_str]

# 取第 2 列开始的数据（跳过日期列）
loads = pd.to_numeric(load_row.iloc[0, 1:], errors='coerce').fillna(0).values.astype(float)[:144]
pvs = pd.to_numeric(pv_row.iloc[0, 1:], errors='coerce').fillna(0).values.astype(float)[:144]
prices = pd.to_numeric(price_row.iloc[0, 1:], errors='coerce').fillna(0).values.astype(float)[:144]

print(f"日期: {date_str}")
print(f"前3个负载: {loads[:3]}")   
print(f"前3个光伏: {pvs[:3]}")     
print(f"前3个电价: {prices[:3]}")

T = 144
dt = 1/6

# 2. 建立问题4-2模型
prob = pulp.LpProblem("Microgrid_Q4_2", pulp.LpMinimize)
buy = pulp.LpVariable.dicts("buy", range(T), lowBound=0)
emergency = pulp.LpVariable.dicts("emergency", range(T), lowBound=0)
curtail = pulp.LpVariable.dicts("curtail", range(T), lowBound=0)
charge = pulp.LpVariable.dicts("charge", range(T), lowBound=0, upBound=5000/6)
discharge = pulp.LpVariable.dicts("discharge", range(T), lowBound=0, upBound=5000/6)
soc = pulp.LpVariable.dicts("soc", range(T+1), lowBound=1200, upBound=10800)

prob += pulp.lpSum([prices[t] * buy[t] + 5 * prices[t] * emergency[t] for t in range(T)])

for t in range(T):
    prob += buy[t] + emergency[t] + (pvs[t] * dt - curtail[t]) + discharge[t] * dt == loads[t] * dt + charge[t] * dt
    prob += soc[t+1] == soc[t] + 0.9 * charge[t] - (1/0.9) * discharge[t]

prob += soc[0] == 6000
prob += soc[T] >= 5900
prob += soc[T] <= 6100

prob.solve()
print("\n求解状态:", pulp.LpStatus[prob.status])

if pulp.LpStatus[prob.status] == 'Optimal':
    print(f"最优全天购电费用: {pulp.value(prob.objective):.2f} 元")
    total_emergency = sum([emergency[t].varValue for t in range(T)])
    print(f"全天紧急购电量: {total_emergency:.2f} kWh")
    for t in range(5):
        print(f"时间 {t+1}: 计划买电 {buy[t].varValue:.2f}, 储能 {soc[t].varValue:.2f}")

日期: 2025-04-15
前3个负载: [3393.4435 3422.0494 3326.52  ]
前3个光伏: [0. 0. 0.]
前3个电价: [0.456  0.4383 0.3894]
Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /usr/local/lib/python3.12/dist-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/230c8952ac3d47c3a4c8537616f1fd2b-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /tmp/230c8952ac3d47c3a4c8537616f1fd2b-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 296 COLUMNS
At line 1884 RHS
At line 2176 BOUNDS
At line 2755 ENDATA
Problem MODEL has 291 rows, 865 columns and 1299 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 149 (-142) rows, 386 (-479) columns and 549 (-750) elements
0  Obj 45536.67 Primal inf 10540.151 (9) Dual inf 11.559811 (104)
72  Obj 39839.925 Primal inf 64362.444 (66)
149  Obj 43134.475 Primal inf 80.273821 (3)
153  Obj 43148.14
Optimal - objective value 43148.14
After Postsolve,

In [25]:
import pandas as pd
import pulp
import numpy as np

# 1. 读取数据
xls = pd.ExcelFile('/kaggle/input/datasets/ericlin073233/program/2.xlsx')
df_load = pd.read_excel(xls, sheet_name=0, header=0)   # 第1张表：负载
df_pv = pd.read_excel(xls, sheet_name=1, header=0)     # 第2张表：光伏
df_price_wave = pd.read_excel('/kaggle/input/datasets/ericlin073233/program/4.xlsx', header=0)

for df in [df_load, df_pv, df_price_wave]:
    df['日期'] = pd.to_datetime(df.iloc[:, 0]).dt.strftime('%Y-%m-%d')

all_dates = df_load['日期'].unique()
target_dates = [d for d in all_dates if '2025-04-11' <= d <= '2025-12-31']
print(f"共需处理 {len(target_dates)} 天...")

T = 144
dt = 1/6

buy_records = []
cd_records = []
emergency_records = []

for i, date_str in enumerate(target_dates):
    if i % 20 == 0:
        print(f"进度: {i+1}/{len(target_dates)} - {date_str}")

    load_row = df_load[df_load['日期'] == date_str]
    pv_row = df_pv[df_pv['日期'] == date_str]
    price_row = df_price_wave[df_price_wave['日期'] == date_str]
    
    if load_row.empty or pv_row.empty or price_row.empty:
        continue
    
    loads = pd.to_numeric(load_row.iloc[0, 1:], errors='coerce').fillna(0).values.astype(float)[:144]
    pvs = pd.to_numeric(pv_row.iloc[0, 1:], errors='coerce').fillna(0).values.astype(float)[:144]
    prices = pd.to_numeric(price_row.iloc[0, 1:], errors='coerce').fillna(0).values.astype(float)[:144]

    # 跳过全0数据
    if np.sum(loads) == 0 and np.sum(pvs) == 0:
        continue

    try:
        prob = pulp.LpProblem(f"Q4_2_{date_str}", pulp.LpMinimize)
        buy = pulp.LpVariable.dicts("buy", range(T), lowBound=0)
        emergency = pulp.LpVariable.dicts("emergency", range(T), lowBound=0)
        curtail = pulp.LpVariable.dicts("curtail", range(T), lowBound=0)
        charge = pulp.LpVariable.dicts("charge", range(T), lowBound=0, upBound=5000/6)
        discharge = pulp.LpVariable.dicts("discharge", range(T), lowBound=0, upBound=5000/6)
        soc = pulp.LpVariable.dicts("soc", range(T+1), lowBound=1200, upBound=10800)

        prob += pulp.lpSum([prices[t] * buy[t] + 5 * prices[t] * emergency[t] for t in range(T)])

        for t in range(T):
            prob += buy[t] + emergency[t] + (pvs[t] * dt - curtail[t]) + discharge[t] * dt == loads[t] * dt + charge[t] * dt
            prob += soc[t+1] == soc[t] + 0.9 * charge[t] - (1/0.9) * discharge[t]

        prob += soc[0] == 6000
        prob += soc[T] >= 5900
        prob += soc[T] <= 6100
        prob.solve()

        if pulp.LpStatus[prob.status] != 'Optimal':
            continue

        for t in range(T):
            time_str = f"{(t//6):02d}:{(t%6)*10:02d}:00"
            buy_records.append({'日期': date_str, '时间段': time_str, '购电量(kWh)': buy[t].varValue})
            cd_records.append({
                '日期': date_str, '时间段': time_str,
                '充电量(kWh)': charge[t].varValue,
                '放电量(kWh)': discharge[t].varValue,
                '0:00储电量(kWh)': soc[0].varValue if t == 0 else None,
                '24:00储电量(kWh)': soc[T].varValue if t == T - 1 else None
            })
            if emergency[t].varValue > 0.01:
                emergency_records.append({'日期': date_str, '时间段': time_str, '紧急购电量(kWh)': emergency[t].varValue})

    except Exception as e:
        print(f"❌ {date_str} 出错: {e}")
        continue

# 读出 result4-2.xlsx
with pd.ExcelWriter('result4-2.xlsx') as writer:
    pd.DataFrame(buy_records).to_excel(writer, sheet_name='计划购电量', index=False)
    pd.DataFrame(cd_records).to_excel(writer, sheet_name='充放电量', index=False)
    if len(emergency_records) > 0:
        pd.DataFrame(emergency_records).to_excel(writer, sheet_name='紧急购电量', index=False)
    else:
        pd.DataFrame(columns=['日期', '时间段', '紧急购电量(kWh)']).to_excel(writer, sheet_name='紧急购电量', index=False)

print(f"\n✅ result4-2.xlsx 已生成！总计 {len(buy_records)//144} 天。")

共需处理 265 天...
进度: 1/265 - 2025-04-11
Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /usr/local/lib/python3.12/dist-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/1ade34903f7b48208602ca771ba381b3-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /tmp/1ade34903f7b48208602ca771ba381b3-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 296 COLUMNS
At line 1884 RHS
At line 2176 BOUNDS
At line 2755 ENDATA
Problem MODEL has 291 rows, 865 columns and 1299 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 147 (-144) rows, 338 (-527) columns and 497 (-802) elements
0  Obj 20197.775 Primal inf 11283.281 (10) Dual inf 7.279282 (83)
66  Obj 16525.224 Primal inf 54867.988 (56)
127  Obj 18534.094
Optimal - objective value 18534.094
After Postsolve, objective 18534.094, infeasibilities - dual 0.0572699 (1), primal 0 (0)
Presolved model was optimal, full 

In [26]:
import pandas as pd
import pulp
import numpy as np

# 1. 读取数据
xls = pd.ExcelFile('/kaggle/input/datasets/ericlin073233/program/2.xlsx')
df_load = pd.read_excel(xls, sheet_name=0, header=0)   # 第1张表：负载
df_pv = pd.read_excel(xls, sheet_name=1, header=0)     # 第2张表：光伏
df_price_wave = pd.read_excel('/kaggle/input/datasets/ericlin073233/program/4.xlsx', header=0)

# 统一日期格式
for df in [df_load, df_pv, df_price_wave]:
    df['日期'] = pd.to_datetime(df.iloc[:, 0]).dt.strftime('%Y-%m-%d')

all_dates = df_load['日期'].unique()
target_dates = [d for d in all_dates if '2025-04-11' <= d <= '2025-12-31']
print(f"共需处理 {len(target_dates)} 天...")

T = 144
dt = 1/6
np.random.seed(42)

plan_records = []
adjust_records = []
cd_records = []
emergency_records = []

for i, date_str in enumerate(target_dates):
    if i % 20 == 0:
        print(f"进度: {i+1}/{len(target_dates)} - {date_str}")

    load_row = df_load[df_load['日期'] == date_str]
    pv_row = df_pv[df_pv['日期'] == date_str]
    price_row = df_price_wave[df_price_wave['日期'] == date_str]
    
    if load_row.empty or pv_row.empty or price_row.empty:
        continue
    
    loads = pd.to_numeric(load_row.iloc[0, 1:], errors='coerce').fillna(0).values.astype(float)[:144]
    pvs_actual = pd.to_numeric(pv_row.iloc[0, 1:], errors='coerce').fillna(0).values.astype(float)[:144]
    prices = pd.to_numeric(price_row.iloc[0, 1:], errors='coerce').fillna(0).values.astype(float)[:144]

    # 跳过全 0 数据
    if np.sum(loads) == 0 and np.sum(pvs_actual) == 0:
        continue

    # 模拟 0:00 的预测误差（30%）
    forecast_0 = np.clip(pvs_actual + np.random.normal(0, 0.30, T), 0, None)

    try:
        # 阶段一：0:00 计划（基于有误差的预测） 
        prob_plan = pulp.LpProblem(f"Plan_{date_str}", pulp.LpMinimize)
        plan_buy = pulp.LpVariable.dicts("plan_buy", range(T), lowBound=0)
        plan_charge = pulp.LpVariable.dicts("plan_charge", range(T), lowBound=0, upBound=5000/6)
        plan_discharge = pulp.LpVariable.dicts("plan_discharge", range(T), lowBound=0, upBound=5000/6)
        plan_soc = pulp.LpVariable.dicts("plan_soc", range(T+1), lowBound=1200, upBound=10800)

        prob_plan += pulp.lpSum([prices[t] * plan_buy[t] for t in range(T)])
        for t in range(T):
            prob_plan += plan_buy[t] + (forecast_0[t] * dt) + plan_discharge[t] * dt == loads[t] * dt + plan_charge[t] * dt
            prob_plan += plan_soc[t+1] == plan_soc[t] + 0.9 * plan_charge[t] - (1/0.9) * plan_discharge[t]
        prob_plan += plan_soc[0] == 6000
        prob_plan += plan_soc[T] >= 5900
        prob_plan += plan_soc[T] <= 6100
        prob_plan.solve()
        plan_buy_values = [plan_buy[t].varValue if plan_buy[t].varValue else 0 for t in range(T)]

        #  阶段二：调整（基于真实数据，产生违约金） 
        prob_adjust = pulp.LpProblem(f"Adjust_{date_str}", pulp.LpMinimize)
        adjust_buy = pulp.LpVariable.dicts("adjust_buy", range(T), lowBound=0)
        emergency = pulp.LpVariable.dicts("emergency", range(T), lowBound=0)
        curtail = pulp.LpVariable.dicts("curtail", range(T), lowBound=0)
        adjust_charge = pulp.LpVariable.dicts("adjust_charge", range(T), lowBound=0, upBound=5000/6)
        adjust_discharge = pulp.LpVariable.dicts("adjust_discharge", range(T), lowBound=0, upBound=5000/6)
        adjust_soc = pulp.LpVariable.dicts("adjust_soc", range(T+1), lowBound=1200, upBound=10800)
        over_buy = pulp.LpVariable.dicts("over_buy", range(T), lowBound=0)
        under_buy = pulp.LpVariable.dicts("under_buy", range(T), lowBound=0)

        prob_adjust += pulp.lpSum([
            prices[t] * plan_buy_values[t] + 5 * prices[t] * emergency[t] +
            0.5 * prices[t] * over_buy[t] + 1.5 * prices[t] * under_buy[t]
            for t in range(T)
        ])
        for t in range(T):
            prob_adjust += adjust_buy[t] + emergency[t] + (pvs_actual[t] * dt - curtail[t]) + adjust_discharge[t] * dt == loads[t] * dt + adjust_charge[t] * dt
            prob_adjust += adjust_soc[t+1] == adjust_soc[t] + 0.9 * adjust_charge[t] - (1/0.9) * adjust_discharge[t]
            prob_adjust += adjust_buy[t] - plan_buy_values[t] == under_buy[t] - over_buy[t]
        prob_adjust += adjust_soc[0] == 6000
        prob_adjust += adjust_soc[T] >= 5900
        prob_adjust += adjust_soc[T] <= 6100
        prob_adjust.solve()

        if pulp.LpStatus[prob_adjust.status] != 'Optimal':
            continue

        #  记录结果 
        for t in range(T):
            time_str = f"{(t//6):02d}:{(t%6)*10:02d}:00"
            plan_records.append({'日期': date_str, '时间段': time_str, '计划购电量(kWh)': plan_buy_values[t]})
            adjust_records.append({'日期': date_str, '时间段': time_str, '调整购电量(kWh)': adjust_buy[t].varValue})
            cd_records.append({
                '日期': date_str, '时间段': time_str,
                '充电量(kWh)': adjust_charge[t].varValue,
                '放电量(kWh)': adjust_discharge[t].varValue,
                '0:00储电量(kWh)': adjust_soc[0].varValue if t == 0 else None,
                '24:00储电量(kWh)': adjust_soc[T].varValue if t == T - 1 else None
            })
            if emergency[t].varValue > 0.01:
                emergency_records.append({'日期': date_str, '时间段': time_str, '紧急购电量(kWh)': emergency[t].varValue})

    except Exception as e:
        print(f"❌ {date_str} 出错: {e}")
        continue

#  读出 result4-3.xlsx 
with pd.ExcelWriter('result4-3.xlsx') as writer:
    pd.DataFrame(plan_records).to_excel(writer, sheet_name='计划购电量', index=False)
    pd.DataFrame(adjust_records).to_excel(writer, sheet_name='调整购电量', index=False)
    pd.DataFrame(cd_records).to_excel(writer, sheet_name='充放电量', index=False)
    if len(emergency_records) > 0:
        pd.DataFrame(emergency_records).to_excel(writer, sheet_name='紧急购电量', index=False)
    else:
        pd.DataFrame(columns=['日期', '时间段', '紧急购电量(kWh)']).to_excel(writer, sheet_name='紧急购电量', index=False)

print(f"\n✅ result4-3.xlsx 已生成！总计 {len(plan_records)//144} 天。")

共需处理 265 天...
进度: 1/265 - 2025-04-11
Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /usr/local/lib/python3.12/dist-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/3adc47a6c2b74ed2822ca6ab35ffe85b-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /tmp/3adc47a6c2b74ed2822ca6ab35ffe85b-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 296 COLUMNS
At line 1452 RHS
At line 1744 BOUNDS
At line 2323 ENDATA
Problem MODEL has 291 rows, 577 columns and 1011 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve determined that the problem was infeasible with tolerance of 1e-08
Analysis indicates model infeasible or unbounded
Perturbing problem by 0.001% of 0.3646006 - largest nonzero change 1.328471e-07 ( 0.00028007595%) - largest zero change 1.3281278e-07
0  Obj 0.013165348 Primal inf 212861.83 (146)
80  Obj 1947.6619 Primal inf 668917.86 (188)
99  Obj